# EDA 004.04 — Descriptive Statistics

**Summarising data with numbers before drawing plots**

## Learning Objectives
By the end of this notebook you will be able to:
- Calculate and interpret measures of central tendency (mean, median, mode)
- Quantify spread using std, variance, IQR, and coefficient of variation
- Measure distribution shape with skewness and kurtosis
- Use percentiles and quantiles to describe data position
- Extend `pandas.describe()` for a richer summary
- Identify common distribution shapes (normal, skewed, bimodal)

---

## Why Descriptive Statistics?

Visualisations show patterns; descriptive statistics **quantify** them precisely and are reproducible across team members, reports, and automated pipelines.

> *"If you cannot measure it, you cannot improve it."* — Lord Kelvin

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

tips = sns.load_dataset('tips')
diamonds = sns.load_dataset('diamonds')

print('Tips:', tips.shape)
tips.head()

---
## 1 — Measures of Central Tendency

**Reference:** [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

| Measure | Formula | Use when |
|---|---|---|
| **Mean** | $\bar{x} = \frac{1}{n}\sum x_i$ | Symmetric distributions, no extreme outliers |
| **Median** | Middle value | Skewed distributions or data with outliers |
| **Mode** | Most frequent value | Categorical data; bimodal distributions |

> **Key insight:** When mean >> median, the distribution is right-skewed (outliers pull the mean up).

In [ ]:
bill = tips['total_bill']

mean_v   = bill.mean()
median_v = bill.median()
mode_v   = bill.mode()[0]

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(bill, bins=30, kde=True, ax=ax, color='steelblue', alpha=0.6)
ax.axvline(mean_v,   color='red',    linestyle='--', lw=2, label=f'Mean = {mean_v:.2f}')
ax.axvline(median_v, color='green',  linestyle='-',  lw=2, label=f'Median = {median_v:.2f}')
ax.axvline(mode_v,   color='orange', linestyle=':',  lw=2, label=f'Mode ≈ {mode_v:.2f}')
ax.legend()
ax.set_title('Total Bill — Mean vs Median vs Mode')
plt.tight_layout()
plt.show()

print(f'Mean:   {mean_v:.2f}')
print(f'Median: {median_v:.2f}')
print(f'Mean > Median → right-skewed distribution')

---
## 2 — Measures of Spread

**Reference:** [numpy statistics](https://numpy.org/doc/stable/reference/routines.statistics.html)

| Measure | Formula / Description | Robust to outliers? |
|---|---|---|
| **Range** | max − min | No |
| **Variance** | $s^2 = \frac{\sum(x_i - \bar{x})^2}{n-1}$ | No |
| **Std deviation** | $s = \sqrt{s^2}$ (same units as data) | No |
| **IQR** | Q3 − Q1 | Yes |
| **Coefficient of Variation** | $CV = s / \bar{x}$ (relative spread) | No |

> Use **IQR** when there are outliers. Use **CV** to compare spread of variables on different scales.

In [ ]:
from scipy.stats import iqr

for col in ['total_bill', 'tip', 'size']:
    s = tips[col]
    print(f"--- {col} ---")
    print(f"  Range : {s.max() - s.min():.2f}")
    print(f"  Std   : {s.std():.2f}")
    print(f"  IQR   : {iqr(s):.2f}")
    print(f"  CV    : {s.std() / s.mean():.3f}  ({s.std() / s.mean() * 100:.1f}%)")
    print()

# Visualise spread across groups
fig, ax = plt.subplots(figsize=(9, 4))
# Show IQR and whiskers clearly
sns.boxplot(data=tips, x='day', y='total_bill', palette='Set2',
            order=['Thur', 'Fri', 'Sat', 'Sun'], ax=ax,
            width=0.5, fliersize=5)
ax.set_title('Spread of Bill by Day (IQR = box, whiskers = 1.5×IQR)')
plt.tight_layout()
plt.show()

---
## 3 — Skewness and Kurtosis

**Reference:** [scipy.stats.skew](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skew.html) | [scipy.stats.kurtosis](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kurtosis.html)

### Skewness — asymmetry of the distribution

| Skewness value | Shape |
|---|---|
| ≈ 0 | Symmetric |
| > 0 (positive) | Right-skewed (long right tail) |
| < 0 (negative) | Left-skewed (long left tail) |

### Kurtosis — heaviness of the tails

| Kurtosis (excess) | Shape | Name |
|---|---|---|
| ≈ 0 | Normal tails | Mesokurtic |
| > 0 | Heavier tails than normal | Leptokurtic |
| < 0 | Lighter tails than normal | Platykurtic |

In [ ]:
np.random.seed(42)

# Construct three distributions with different skewness
symmetric    = np.random.normal(0, 1, 1000)
right_skewed = np.random.exponential(scale=1, size=1000)
left_skewed  = -np.random.exponential(scale=1, size=1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, label, color in zip(
    axes,
    [symmetric, right_skewed, left_skewed],
    ['Symmetric (Normal)', 'Right-Skewed (Exponential)', 'Left-Skewed'],
    ['steelblue', 'coral', 'seagreen']
):
    sk = stats.skew(data)
    ku = stats.kurtosis(data)  # excess kurtosis
    sns.histplot(data, bins=40, kde=True, ax=ax, color=color, alpha=0.6)
    ax.axvline(np.mean(data), color='red', linestyle='--', lw=1.5, label='Mean')
    ax.axvline(np.median(data), color='black', linestyle='-', lw=1.5, label='Median')
    ax.set_title(f'{label}\nskewness={sk:.2f}, kurtosis={ku:.2f}')
    ax.legend(fontsize=8)

plt.suptitle('Skewness and Kurtosis', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nReal data examples:')
for col in ['total_bill', 'tip']:
    sk = stats.skew(tips[col])
    ku = stats.kurtosis(tips[col])
    print(f'  {col}: skewness={sk:.3f}, kurtosis={ku:.3f}')

---
## 4 — Percentiles and Quantiles

**Reference:** [numpy.percentile](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html)

The **$p$-th percentile** is the value below which $p$% of the data falls.

| Percentile | Alias | Meaning |
|---|---|---|
| 25th | Q1 | 25% of data is below this |
| 50th | Q2, Median | 50% below |
| 75th | Q3 | 75% below |
| 90th | P90 | 90% below |
| 99th | P99 | 99% below (outlier threshold) |

Percentiles are the basis of the **box plot** and are the backbone of **quantile-based features**.

In [ ]:
bill = tips['total_bill']

pcts = [10, 25, 50, 75, 90, 95, 99]
values = np.percentile(bill, pcts)

print("Percentiles of Total Bill:")
for p, v in zip(pcts, values):
    print(f"  P{p:2d}: ${v:.2f}")

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(bill, bins=30, kde=True, ax=ax, color='steelblue', alpha=0.5)

colors = ['orange', 'red', 'green', 'red', 'orange', 'purple', 'purple']
for p, v, c in zip(pcts, values, colors):
    ax.axvline(v, color=c, linestyle=':', alpha=0.8, lw=1.5)
    ax.text(v + 0.3, ax.get_ylim()[1] * 0.95, f'P{p}', fontsize=8, color=c)

ax.set_title('Total Bill — Percentile Lines')
plt.tight_layout()
plt.show()

---
## 5 — Extended pandas describe()

**Reference:** [pandas.DataFrame.describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)

`describe()` provides a quick statistical summary, but you can extend it to include:
- Additional percentiles (e.g. P1, P99)
- Skewness and kurtosis
- Missing value count
- Coefficient of variation

In [ ]:
def extended_describe(df, numeric_only=True):
    """Extended summary beyond pandas describe()."""
    num_df = df.select_dtypes(include='number') if numeric_only else df
    base = num_df.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    base['skewness'] = num_df.skew()
    base['kurtosis'] = num_df.kurtosis()
    base['missing']  = num_df.isna().sum()
    base['cv']       = num_df.std() / num_df.mean()
    return base.round(3)

extended_describe(tips)

---
## 6 — Distribution Shapes Reference

Recognising distribution shape guides preprocessing and model selection:

| Shape | Example data | skewness | Typical fix |
|---|---|---|---|
| **Normal** | Heights, IQ scores | ≈ 0 | None needed |
| **Right-skewed** | Income, house prices, fare | > 1 | Log or sqrt transform |
| **Left-skewed** | Exam scores near max | < −1 | Power transform |
| **Bimodal** | Mixture of two populations | varies | Investigate + segment |
| **Uniform** | Random numbers | ≈ 0 | No distribution assumption |
| **Heavy-tailed** | Financial returns | High kurtosis | Robust methods |

---
## Key Takeaways

- **Mean** is sensitive to outliers; prefer **median** for skewed data
- **IQR** is a robust measure of spread; **CV** allows comparison across variables on different scales
- **Skewness > 0** means right tail is longer (common in income, prices)
- **High kurtosis** means more extreme outliers than a normal distribution would have
- Always check P1 and P99 — values at the extremes often reveal data quality issues

---
## Further Reading

- [scipy.stats documentation](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [Pandas — Descriptive Statistics](https://pandas.pydata.org/docs/user_guide/basics.html#descriptive-statistics)
- [Think Stats (free online)](https://greenteapress.com/wp/think-stats-2e/) — Allen B. Downey
- [Statistics Done Wrong (free online)](https://www.statisticsdonewrong.com/) — Alex Reinhart
- [Practical Statistics for Data Scientists](https://www.oreilly.com/library/view/practical-statistics-for/9781492072935/) — Bruce & Bruce